# AskGeorge — Building My AI Representative

This notebook builds **AskGeorge**, a chat assistant that answers questions from recruiters and hiring managers *as George Traskas*, grounded strictly in his real professional background.

The build has five steps:
1. **Ground** an LLM (any model via OpenRouter) in George's background (LinkedIn profile, summary, project portfolio from `askgeorge/me/`).
2. **Test** it on realistic recruiter questions.
3. **Stream** the reply token by token, like a real chat product.
4. **Add a quality gate** — a second model evaluates every draft answer and forces a retry when it is off-brand or ungrounded.
5. **Add tools** — unknown questions and interested visitors are emailed straight to George.

The packaged, deployable version lives in [`askgeorge/`](askgeorge/) (Gradio app + Modal deployment).

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel

load_dotenv(override=True)

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
CHAT_MODEL = os.getenv("OPENROUTER_MODEL", "google/gemini-3.1-flash-lite")
EVAL_MODEL = "google/gemini-3.1-flash-lite"
# Thinking models (Gemini 3.x, GPT-5 family) reason silently before writing;
# capping the effort keeps first tokens fast.
REASONING = {"reasoning": {"effort": "low"}}

api_key = os.getenv("OPENROUTER_API_KEY")
assert api_key, "Set OPENROUTER_API_KEY in a .env file (get one at openrouter.ai/keys)"
client = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=api_key)
print(f"OpenRouter client ready — chat model: {CHAT_MODEL}")

## 1. Ground the model in the real background

Three Markdown documents define what the assistant is allowed to claim:
- `linkedin.md` — full LinkedIn profile
- `summary.md` — distilled professional summary
- `projects.md` — project portfolio deep dive with design decisions and Q&A

In [ ]:
DATA_DIR = Path("askgeorge") / "me"

background = "\n\n".join(
    f"## {name}\n\n{(DATA_DIR / name).read_text(encoding='utf-8')}"
    for name in ("summary.md", "linkedin.md", "projects.md")
)

SYSTEM_PROMPT = f"""You are acting as Georgios (George) Traskas, answering questions on his
professional page. Speak in the first person, as George, to visitors such as
recruiters, hiring managers, and potential clients.

Rules:
- Be professional, warm, and concise; answer like a confident engineer in a friendly interview.
- Ground every claim strictly in the background below. Never invent employers, dates, metrics, or skills.
- If a question cannot be answered from the background, say so honestly.
- For follow-ups, share George's email (georgiost77@gmail.com), LinkedIn
  (linkedin.com/in/george-traskas), and GitHub (github.com/gtraskas).
- Politely decline topics unrelated to George's professional profile.

# Background

{background}"""

print(f"Background loaded: {len(background):,} characters")

In [ ]:
def ask(question: str) -> str:
    """Return a grounded first-person answer to a visitor question."""
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question},
        ],
    )
    return response.choices[0].message.content or ""

print(ask("What is your experience with RAG in production?"))

## 2. Streaming — tokens as they arrive

A recruiter should never stare at a spinner. With `stream=True` the API returns
chunks as the model generates them; the packaged Gradio app yields the growing
text so the reply types itself out in the UI.

In [ ]:
def ask_streaming(question: str) -> None:
    """Print the answer token by token as it streams from the model."""
    stream = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question},
        ],
        stream=True,
    )
    for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content:
            print(chunk.choices[0].delta.content, end="", flush=True)
    print()

ask_streaming("Tell me about your most recent project.")

## 3. Quality gate — evaluate every answer before it ships

A recruiter-facing assistant cannot afford a sloppy or hallucinated reply. A second, cheaper model grades each draft against the same background; rejected drafts are regenerated with the evaluator's feedback injected.

In [ ]:
class Evaluation(BaseModel):
    """Verdict on a drafted answer."""

    is_acceptable: bool
    feedback: str


def evaluate(question: str, draft_answer: str) -> Evaluation:
    """Grade a draft: professional tone, first person, strictly grounded."""
    evaluator_prompt = (
        "You evaluate an AI agent that answers as George Traskas on his professional page. "
        "Reject the answer if it invents facts not in the background, breaks first person, "
        "or is unprofessional. Reply with your verdict and short feedback.\n\n"
        f"# Background\n\n{background}"
    )
    completion = client.beta.chat.completions.parse(
        model=EVAL_MODEL,
        messages=[
            {"role": "system", "content": evaluator_prompt},
            {
                "role": "user",
                "content": f"Question: {question}\n\nDraft answer: {draft_answer}",
            },
        ],
        response_format=Evaluation,
    )
    parsed = completion.choices[0].message.parsed
    assert parsed is not None
    return parsed


def ask_with_quality_gate(question: str, max_retries: int = 2) -> str:
    """Answer a question, regenerating until the evaluator accepts it."""
    answer = ask(question)
    for attempt in range(max_retries):
        verdict = evaluate(question, answer)
        if verdict.is_acceptable:
            print(f"[evaluator] accepted on attempt {attempt + 1}")
            return answer
        print(f"[evaluator] rejected: {verdict.feedback}")
        retry_prompt = (
            f"{question}\n\n(Your previous draft was rejected by a reviewer with this "
            f"feedback, address it: {verdict.feedback})"
        )
        answer = ask(retry_prompt)
    return answer


print(ask_with_quality_gate("Do you hold a patent?"))

In [ ]:
recruiter_questions = [
    "Why should we hire you as an AI engineer?",
    "Tell me about a production failure you caught and fixed.",
    "Are you open to remote roles, and what is your availability?",
]

for question in recruiter_questions:
    print(f"Q: {question}\n")
    print(f"A: {ask_with_quality_gate(question)}\n")
    print("-" * 80)

## 4. Tools — never lose a lead, never fake an answer

The deployed agent has two tools, wired to **Gmail SMTP** so George gets an email immediately (nothing is stored on the server's ephemeral disk):

- `record_unknown_question` — fires whenever a question cannot be answered from the background, so George can follow up personally.
- `record_contact_request` — fires when a visitor shares their email and interest.

The full streaming tool-calling loop (schemas, dispatch, sanitized history, retry cap, email notifier) is packaged in [`askgeorge/app.py`](askgeorge/app.py) — below we launch that production app directly.

In [ ]:
from askgeorge.app import build_demo

demo = build_demo()
demo.launch()

## 5. Deploy

The app ships to Modal as a CPU container that scales to zero (effectively free at recruiter traffic volumes):

```bash
uv run modal setup   # once
uv run modal secret create askgeorge-secret OPENROUTER_API_KEY=... GMAIL_ADDRESS=... GMAIL_APP_PASSWORD=...
uv run modal deploy askgeorge/deploy_modal.py
```

Live URL: `https://<modal-username>--askgeorge-web.modal.run` — see [`askgeorge/README.md`](askgeorge/README.md) for details.